In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# LEVEL-SPACE STRUCTURAL BASELINE FOR FIVE-DAY BLOCK MINIMUM RETURNS
#
# Code/KAN_Section/Dense_vs_Sparse_KAN/04_volatility_inspired_baseline.ipynb
#
# Standalone. Reads only the two target files, since every regressor is a
# function of returns. Does not touch the feature pipeline, lib/, or any model
# code. Splits are hard-coded to match the KAN exactly.
#
# THE MODEL
#   Y_t = sigma_t * m5_t,   sigma_t ~ sqrt(RV22_t),   m5_t = a0 + a1*acc + a2*asym
#
#   =>  Y_t = a0*sqrt(RV22) + a1*sqrt(RV22)*acc + a2*sqrt(RV22)*asym
#
# Predicts Y directly in return units. No log transform, no truncation on the
# sign of Y, no retransformation. Minimises the same squared-error loss the KAN
# minimises, on the same rows, in the same units — so MSE is directly comparable.
#
# STRUCTURAL PREDICTION
#   RV22 is a 22-day SUM, so sqrt(RV22) = sqrt(22)*sigma_daily = 4.69*sigma_daily.
#   With E[min of 5 standard normals] = -1.163,
#         a0 = -1.163 / 4.69 = -0.248
#   Fat tails push it more negative; errors-in-variables pulls it toward zero.
#   Checked before any result is read.
# ═══════════════════════════════════════════════════════════════════════════════

# ── CELL 1 : CONFIG ───────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 50)
TGT = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets')
RESULTS_DIR = Path('../../../Data/Results/Dense_vs_Sparse_KAN/Volatility_Baseline')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EMBARGO_DAYS = 5      # trading dates dropped from the end of train, matching the KAN
MAX_GAP_DAYS = 10     # calendar gap above which a stock's history is cut into segments
WARMUP = 22           # longest rolling window

SPLITS = {   # chronological: Split_A trains earliest
    'Split_A': {'train_end': '2015-12-31', 'test_start': '2018-01-01', 'test_end': '2019-12-31'},
    'Split_B': {'train_end': '2017-12-31', 'test_start': '2020-01-01', 'test_end': '2021-12-31'},
    'Split_C': {'train_end': '2019-12-31', 'test_start': '2022-01-01', 'test_end': '2023-12-31'},
    'Split_D': {'train_end': '2020-12-31', 'test_start': '2023-01-01', 'test_end': '2024-12-31'},
}

MODELS = {
    'L1': ['sig'],                            # level only
    'L2': ['sig', 'sig_acc'],                 # + short-horizon acceleration
    'L3': ['sig', 'sig_acc', 'sig_asym'],     # + downside asymmetry   <- full model
}
FULL = 'L3'
REG = MODELS[FULL]

# ── THE THREE FACTS THAT DETERMINE EVERYTHING BELOW ──────────────────────────
#
# MARKET ALIGNMENT. target_daily_return is PRE-SHIFTED: row t holds the return
#   realised on day t+1. The target minret_5d_pct[t] is 100*min over indices
#   t..t+4, i.e. calendar days t+1..t+5. Building trailing RV from the unshifted
#   series would include index t, which IS day t+1 and the first day of the
#   target window — a one-day leak. We unshift once so ret[t] is the return
#   realised ON day t.
#
# PANEL ALIGNMENT. dlyret is NOT pre-shifted; dlyret[t] is already day t's
#   return. Verified empirically in Cell 8 by testing both conventions against an
#   independent reconstruction, not assumed.
#
# GAPS. The panel is a rebalanced universe, so stocks enter and exit. Both
#   .rolling() and .shift() operate on ROW POSITION, so at a re-entry boundary
#   they splice across a multi-month hole. Every stock's history is therefore cut
#   into contiguous segments and all windowing is grouped on segment.
#
# DTYPES. dlyret may load as nullable Float64. Under that dtype NA > 0 is NA
#   rather than False, so a (rv > 0) guard silently fails and log(0) = -inf can
#   reach the design matrix — and -inf passes notna(). Everything is cast to
#   plain float64 and usability requires finite.
#
# UNITS. Percent throughout, so MSE and R^2 are directly comparable to the KAN,
#   which predicts minret_5d_pct.
#
# NO TRUNCATION. Rows where Y >= 0 (all five forward days non-negative) are
#   RETAINED. The level model is defined there; a log model is not. This removes
#   the selection-on-the-dependent-variable bias of the log specification and
#   means the baseline is scored on exactly the rows the KAN is scored on.

print('=' * 110)
print('LEVEL-SPACE STRUCTURAL BASELINE')
print('=' * 110)
print(f'  targets    : {TGT}')
print(f'  splits     : {list(SPLITS)}   ({EMBARGO_DAYS}-day embargo at end of train)')
print(f'  models     : ' + ',  '.join(f'{k} ({len(v)})' for k, v in MODELS.items())
      + f'   [full model = {FULL}]')
print(f'  gap policy : segments cut at calendar gaps > {MAX_GAP_DAYS} days')
print(f'  model      : Y = a0*sqrt(RV22) + a1*sqrt(RV22)*acc + a2*sqrt(RV22)*asym')


# ── CELL 2 : CONSTRUCTION HELPERS ─────────────────────────────────────────────
def add_segments(df, group_col='permno', date_col='date', max_gap_days=MAX_GAP_DAYS):
    """
    Cut each entity's history into contiguous segments. A new segment begins at
    the entity's first row, or wherever the calendar gap to the previous row
    exceeds max_gap_days.

    Ordinary weekends are 3 calendar days and holiday clusters reach 5-6, so a
    threshold of 10 catches genuine universe exits (months to years) without
    splitting on normal market closures.

    Grouping subsequent rolling and shift operations on seg_id rather than the
    entity id means they operate by row position WITHIN a contiguous stretch and
    cannot splice across a hole.
    """
    gap = df.groupby(group_col)[date_col].diff().dt.days
    new_seg = gap.isna() | (gap > max_gap_days)
    df = df.copy()
    df['segment'] = new_seg.groupby(df[group_col]).cumsum()
    df['seg_id'] = df[group_col].astype(str) + '_' + df['segment'].astype(str)
    return df, gap


def build_regressors(ret_pct, group=None):
    """
    Trailing volatility quantities and the three level-space regressors.
    Every window ends at t inclusive and uses no information after t.

        RV22_t   = sum_{j=0}^{21} r_{t-j}^2          (level)
        RV5_t    = sum_{j=0}^{4}  r_{t-j}^2
        RSneg_t  = sum_{j=0}^{4}  r_{t-j}^2 * 1[r_{t-j} < 0]

        sig      = sqrt(RV22)                        volatility scale
        sig_acc  = sqrt(RV22) * log(RV5/RV22)        scale x acceleration
        sig_asym = sqrt(RV22) * (RSneg/RV5)          scale x downside asymmetry

    The interactions are not a modelling flourish: the decomposition
    Y = sigma * (a0 + a1*acc + a2*asym) multiplies out to exactly these terms.

    `group` should be a SEGMENT id, not an entity id, so windows cannot span a
    gap in the entity's history.
    """
    ret_pct = pd.Series(np.asarray(ret_pct, dtype='float64'), index=ret_pct.index)
    sq = ret_pct ** 2
    neg = sq.where(ret_pct < 0, 0.0)

    if group is None:
        rv22 = sq.rolling(22).sum()
        rv5 = sq.rolling(5).sum()
        rsn = neg.rolling(5).sum()
    else:
        rv22 = sq.groupby(group).rolling(22).sum().reset_index(level=0, drop=True)
        rv5 = sq.groupby(group).rolling(5).sum().reset_index(level=0, drop=True)
        rsn = neg.groupby(group).rolling(5).sum().reset_index(level=0, drop=True)

    out = pd.DataFrame({'rv22': rv22, 'rv5': rv5, 'rs_neg': rsn}).astype('float64')

    # acc uses log(RV5/RV22), so both must be strictly positive. RV == 0 means
    # 22 (or 5) consecutive days of exactly zero return: impossible for the
    # index, rare but possible for a thinly traded stock or a stale price.
    n_zero22 = int((out['rv22'] == 0).sum())
    n_zero5 = int((out['rv5'] == 0).sum())
    out.loc[~((out['rv22'] > 0) & (out['rv5'] > 0)), :] = np.nan

    s = np.sqrt(out['rv22'])
    out['sig'] = s
    out['sig_acc'] = s * np.log(out['rv5'] / out['rv22'])
    out['sig_asym'] = s * (out['rs_neg'] / out['rv5'])

    out.attrs['n_zero_rv22'] = n_zero22
    out.attrs['n_zero_rv5'] = n_zero5
    return out


def forward_min_5d(ret, group=None):
    """
    Independent reconstruction of the 5-day forward block minimum, plus a mask
    marking rows whose full forward window lies inside one contiguous segment.

    min(axis=1) skips NaN by default, so without the completeness mask a row near
    a segment boundary would silently return a minimum over FEWER than five days
    and could agree or disagree with the stored target by accident.
    """
    if group is None:
        cols = [ret.shift(-k) for k in range(1, 6)]
    else:
        cols = [ret.groupby(group).shift(-k) for k in range(1, 6)]
    fwd = pd.concat(cols, axis=1)
    return fwd.min(axis=1), fwd.notna().all(axis=1)


# ── CELL 3 : MARKET BUILD ─────────────────────────────────────────────────────
print('\n' + '=' * 110)
print('MARKET — BUILD')
print('=' * 110)

m = (pd.read_parquet(TGT / 'targets_market.parquet')
       .sort_values('date').reset_index(drop=True))
m['date'] = pd.to_datetime(m['date'])
m['target_daily_return'] = m['target_daily_return'].astype('float64')
m['minret_5d_pct'] = m['minret_5d_pct'].astype('float64')

# unshift: ret[t] = the return realised ON day t   (see alignment note, Cell 1)
m['ret'] = m['target_daily_return'].shift(1) * 100.0

feat_m = build_regressors(m['ret'])
m = pd.concat([m, feat_m], axis=1)
m['Y'] = m['minret_5d_pct']

m_chk, m['fwd_ok'] = forward_min_5d(m['ret'])

m['usable'] = (m[REG].notna().all(axis=1)
               & np.isfinite(m[REG].to_numpy()).all(axis=1)
               & m['Y'].notna() & m['fwd_ok'])

n_warm = int(m['sig'].notna().sum())
n_pos = int((m['usable'] & (m['Y'] >= 0)).sum())

print(f'  rows                             {len(m):>9,}')
print(f'  after {WARMUP}-day warm-up               {n_warm:>9,}')
print(f'  usable                           {int(m["usable"].sum()):>9,}')
print(f'  dropped, incomplete forward      {int((~m["fwd_ok"]).sum()):>9,}')
print(f'  zero-variance rows               RV22 {feat_m.attrs["n_zero_rv22"]}   '
      f'RV5 {feat_m.attrs["n_zero_rv5"]}')
print(f'  RETAINED with Y >= 0             {n_pos:>9,}   ({n_pos / max(int(m["usable"].sum()), 1):.2%})')
print('    (a log specification must drop these; the level model does not,')
print('     which removes selection on the dependent variable and puts the')
print('     baseline on exactly the rows the KAN is scored on)')


# ── CELL 4 : MARKET CHECKS ────────────────────────────────────────────────────
print('\n' + '=' * 110)
print('CHECKS — MARKET')
print('=' * 110)

# (a) the unshift is arithmetically what we claim
assert np.allclose(m['ret'].iloc[1:] / 100.0, m['target_daily_return'].iloc[:-1],
                   equal_nan=True), 'unshift wrong'
print('  [a] ret[t] == target_daily_return[t-1] * 100                              OK')

# (b) no regressor may correlate with the first day of the target window.
#     r_{t+1} is target_daily_return[t]. A correctly built trailing window cannot
#     see it, so |corr| should be small; a large value means a one-day leak.
sub = m['usable']
worst = max(abs(np.corrcoef(m.loc[sub, c], m.loc[sub, 'target_daily_return'])[0, 1])
            for c in REG)
print(f'  [b] max |corr(regressor, r_t+1)| = {worst:.4f}'
      f'{"                                    OK" if worst < 0.20 else "   <-- INVESTIGATE"}')

# (c) independent reconstruction, complete forward windows only
d = (m_chk - m['Y']).abs()[m['fwd_ok']]
mx = np.nanmax(d)
print(f'  [c] reconstruction of Y (complete windows, n={int(m["fwd_ok"].sum()):,}):'
      f' max abs diff = {mx:.2e}   {"OK" if mx < 1e-8 else "<-- MISMATCH"}')

# (d) RV22 strictly backward looking
i = m.index[m['usable']][500]
manual = (m['ret'].iloc[i - 21:i + 1] ** 2).sum()
print(f'  [d] RV22 at row {i}: manual {manual:.6f} vs stored {m["rv22"].iloc[i]:.6f}   '
      f'{"OK" if abs(manual - m["rv22"].iloc[i]) < 1e-8 else "<-- MISMATCH"}')

MARKET_OK = (worst < 0.20) and (mx < 1e-8)
if not MARKET_OK:
    print('\n  *** MARKET ALIGNMENT CHECK FAILED — do not read the results below ***')


# ── CELL 5 : ESTIMATION MACHINERY ─────────────────────────────────────────────
def ols(X, y):
    """Least squares. Any intercept must already be a column of X."""
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta, y - X @ beta


def cluster_se(X, resid, groups):
    """
    Cluster-robust standard errors.

    Five-day forward windows sampled daily share four of their five days, so
    adjacent residuals are mechanically correlated and classical OLS errors are
    far too small. Clustering by calendar month absorbs that dependence.

    Panel note: this is SINGLE-WAY clustering on month. It absorbs the target's
    5-day overlap and common cross-sectional shocks within a month, but not
    persistent stock-specific serial correlation beyond that window. A fully
    rigorous panel treatment would cluster two-way (stock x month). Stated as a
    simplification for a baseline, not assumed away.
    """
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    codes = pd.factorize(groups)[0]
    meat = np.zeros((k, k))
    for g in np.unique(codes):
        idx = codes == g
        s = X[idx].T @ resid[idx]
        meat += np.outer(s, s)
    G = len(np.unique(codes))
    corr = (G / max(G - 1, 1)) * ((n - 1) / max(n - k, 1))
    V = XtX_inv @ meat @ XtX_inv * corr
    return np.sqrt(np.maximum(np.diag(V), 0.0))


def r2(y, yhat, ref_mean):
    """1 - SSE/SST with the benchmark mean supplied explicitly."""
    sse = np.sum((y - yhat) ** 2)
    sst = np.sum((y - ref_mean) ** 2)
    return 1 - sse / sst if sst > 0 else np.nan


def masks(df, cfg):
    """Train mask with a trading-date embargo, and test mask."""
    d = df['date']
    tr_dates = np.sort(d[d <= cfg['train_end']].unique())
    keep = tr_dates[:-EMBARGO_DAYS] if len(tr_dates) > EMBARGO_DAYS else tr_dates
    return (d.isin(keep) & df['usable'],
            (d >= cfg['test_start']) & (d <= cfg['test_end']) & df['usable'])


def fit_market(df, cols, cfg, split, model, intercept=True):
    tr_m, te_m = masks(df, cfg)
    tr, te = df[tr_m], df[te_m]
    if len(tr) < 50 or len(te) < 20:
        return None

    base_tr = [np.ones(len(tr))] if intercept else []
    base_te = [np.ones(len(te))] if intercept else []
    Xtr = np.column_stack(base_tr + [tr[c].to_numpy() for c in cols])
    Xte = np.column_stack(base_te + [te[c].to_numpy() for c in cols])
    ytr, yte = tr['Y'].to_numpy(), te['Y'].to_numpy()

    beta, resid = ols(Xtr, ytr)
    se = cluster_se(Xtr, resid, tr['date'].dt.to_period('M'))
    yhat = Xte @ beta

    out = {'split': split, 'model': model,
           'test': f"{cfg['test_start'][:4]}-{cfg['test_end'][:4]}",
           'n_fit': len(tr), 'n_test': len(te),
           'R2': r2(yte, yhat, yte.mean()),
           'R2_trainmean': r2(yte, yhat, ytr.mean()),
           'R2_insample': r2(ytr, Xtr @ beta, ytr.mean()),
           'MSE': np.mean((yte - yhat) ** 2),
           'MAE': np.mean(np.abs(yte - yhat)),
           'var_Y': yte.var()}
    for j, nm in enumerate((['const'] if intercept else []) + cols):
        out[f'b_{nm}'] = beta[j]
        out[f't_{nm}'] = beta[j] / se[j] if se[j] > 0 else np.nan
    return out


def fit_panel(df, cols, cfg, split, model):
    """
    Pooled OLS with stock fixed effects via the within transformation: demean y
    and X within stock, fit slopes on the deviations, then recover each fixed
    effect as the group mean of (y - X'beta). Identical to dummy variables and
    far cheaper at 208 stocks.

    Fixed effects are identified on TRAINING data only. A stock appearing in test
    with no training history has no fixed effect and is dropped.

    No intercept column is needed: within-group deviations sum to exactly zero by
    construction, so a constant would be orthogonal to everything.
    """
    tr_m, te_m = masks(df, cfg)
    tr, te = df[tr_m], df[te_m]
    if len(tr) < 500 or len(te) < 100:
        return None

    ytr = tr['Y'].to_numpy()
    Xtr = tr[cols].to_numpy()
    gid = tr['permno'].to_numpy()

    ybar = pd.Series(ytr).groupby(gid).transform('mean').to_numpy()
    Xbar = np.column_stack([pd.Series(Xtr[:, j]).groupby(gid).transform('mean').to_numpy()
                            for j in range(Xtr.shape[1])])
    Xdm, ydm = Xtr - Xbar, ytr - ybar

    beta, _ = ols(Xdm, ydm)
    se = cluster_se(Xdm, ydm - Xdm @ beta, tr['date'].dt.to_period('M'))

    fe = pd.Series(ytr - Xtr @ beta).groupby(gid).mean()
    fitted_tr = fe.reindex(gid).to_numpy() + Xtr @ beta

    has_fe = te['permno'].isin(fe.index).to_numpy()
    te = te[has_fe]
    yte = te['Y'].to_numpy()
    yhat = fe.reindex(te['permno'].to_numpy()).to_numpy() + te[cols].to_numpy() @ beta

    out = {'split': split, 'model': model,
           'test': f"{cfg['test_start'][:4]}-{cfg['test_end'][:4]}",
           'n_fit': len(tr), 'n_test': len(te), 'n_stocks': len(fe),
           'dropped_no_fe': int((~has_fe).sum()),
           'R2': r2(yte, yhat, yte.mean()),
           'R2_trainmean': r2(yte, yhat, ytr.mean()),
           'R2_insample': r2(ytr, fitted_tr, ytr.mean()),
           'MSE': np.mean((yte - yhat) ** 2),
           'MAE': np.mean(np.abs(yte - yhat)),
           'var_Y': yte.var()}
    for j, nm in enumerate(cols):
        out[f'b_{nm}'] = beta[j]
        out[f't_{nm}'] = beta[j] / se[j] if se[j] > 0 else np.nan
    return out


def ladder(res, title):
    """Incremental R^2 across the L1 -> L2 -> L3 ladder, train-mean benchmark."""
    piv = res.pivot(index='split', columns='model', values='R2_trainmean')
    piv['L2-L1'] = piv['L2'] - piv['L1']
    piv['L3-L2'] = piv['L3'] - piv['L2']
    piv['L3-L1'] = piv['L3'] - piv['L1']
    print(f'\n  {title}')
    print('  ' + '-' * 104)
    print(piv.to_string(float_format=lambda v: f'{v:+.4f}'))
    return piv


def headline(res, title, extra=()):
    """One row per split, full model only, with coefficients and t in brackets."""
    t = res[res['model'] == FULL].copy()
    disp = t.copy()
    for nm in REG:
        disp[nm] = t.apply(lambda r: f'{r[f"b_{nm}"]:+.3f} [{r[f"t_{nm}"]:+.1f}]', axis=1)
    cols = (['split', 'test', 'n_fit', 'n_test'] + list(extra) + REG
            + ['R2', 'R2_trainmean', 'MSE', 'MAE', 'var_Y'])
    print('\n' + '=' * 118)
    print(title)
    print('=' * 118)
    print(disp[cols].to_string(index=False, formatters={
        'n_fit': '{:,}'.format, 'n_test': '{:,}'.format,
        'R2': '{:+.4f}'.format, 'R2_trainmean': '{:+.4f}'.format,
        'MSE': '{:.4f}'.format, 'MAE': '{:.4f}'.format, 'var_Y': '{:.4f}'.format}))
    return t


# ── CELL 6 : MARKET RESULTS ───────────────────────────────────────────────────
res_m = pd.DataFrame([r for s, c in SPLITS.items() for mn, cols in MODELS.items()
                      if (r := fit_market(m, cols, c, s, mn)) is not None])

tbl_m = headline(res_m, 'TABLE 1 — MARKET.  Full model (L3), level space, all valid test rows.')
piv_m = ladder(res_m, 'MARKET — incremental R^2 (train-mean benchmark)')


# ── CELL 7 : STRUCTURAL DIAGNOSTIC ON a0 ──────────────────────────────────────
print('\n' + '=' * 110)
print('DIAGNOSTIC ON a0  (coefficient on sqrt(RV22))')
print('=' * 110)
print("""
  Y = sigma * E[min of 5 draws], and RV22 is a 22-day SUM, so
  sqrt(RV22) = sqrt(22) * sigma_daily = 4.69 * sigma_daily. With
  E[min of 5 standard normals] = -1.163,

        a0 = -1.163 / 4.69 = -0.248

  Fat tails push it more negative; errors-in-variables pulls it toward zero.
  Expect roughly [-0.35, -0.20]. Outside [-0.50, -0.10], stop and investigate
  units, sign, or index construction.

  The fitted model carries an intercept, so a0 is strictly "the slope on sigma
  with a free constant" rather than the structural quantity, which assumed
  Y = sigma * m5 with no additive term. The no-intercept column is the clean
  structural test; if it sits closer to -0.248, the decomposition is doing the
  work rather than the constant.
""")

noint = pd.DataFrame([fit_market(m, REG, c, s, FULL, intercept=False)
                      for s, c in SPLITS.items()])

diag = pd.DataFrame({
    'split':         tbl_m['split'].values,
    'a0':            tbl_m['b_sig'].values,
    't':             tbl_m['t_sig'].values,
    'const':         tbl_m['b_const'].values,
    'a0_no_const':   noint['b_sig'].values,
})
diag['verdict'] = np.where(diag['a0'].between(-0.50, -0.10), 'as expected', 'INVESTIGATE')
print(diag.to_string(index=False, float_format=lambda v: f'{v:.3f}'))


# ── CELL 8 : PANEL BUILD ──────────────────────────────────────────────────────
print('\n' + '=' * 110)
print('PANEL — BUILD')
print('=' * 110)

p = (pd.read_parquet(TGT / 'targets_panel.parquet')
       .sort_values(['permno', 'date']).reset_index(drop=True))
p['date'] = pd.to_datetime(p['date'])

# force plain float64: nullable Float64 makes (rv > 0) evaluate to NA rather than
# False, so zero-variance rows escape the mask
p['ret'] = p['dlyret'].astype('float64') * 100.0
p['minret_5d_pct'] = p['minret_5d_pct'].astype('float64')

p, gap = add_segments(p)

print('  calendar gap distribution within stock (days, top 12):')
print(gap.value_counts().sort_index().head(12).to_string())
print(f'\n  gaps > {MAX_GAP_DAYS} days              {int((gap > MAX_GAP_DAYS).sum()):>9,}')
print(f'  segments                   {p["seg_id"].nunique():>9,}   across {p["permno"].nunique()} stocks')

seg_len = p.groupby('seg_id').size()
print(f'  segments < {WARMUP} rows           {int((seg_len < WARMUP).sum()):>9,}')

feat_p = build_regressors(p['ret'], group=p['seg_id'])
print(f'  zero-variance rows         RV22 {feat_p.attrs["n_zero_rv22"]:,}   '
      f'RV5 {feat_p.attrs["n_zero_rv5"]:,}')
p = pd.concat([p, feat_p], axis=1)
p['Y'] = p['minret_5d_pct']

p_chk, p['fwd_ok'] = forward_min_5d(p['ret'], group=p['seg_id'])

p['usable'] = (p[REG].notna().all(axis=1)
               & np.isfinite(p[REG].to_numpy()).all(axis=1)
               & p['Y'].notna() & p['fwd_ok'])

n_pos_p = int((p['usable'] & (p['Y'] >= 0)).sum())
print(f'\n  rows                       {len(p):>9,}')
print(f'  usable                     {int(p["usable"].sum()):>9,}')
print(f'  dropped, window spans gap  {int((~p["fwd_ok"]).sum()):>9,}   ({(~p["fwd_ok"]).mean():.3%})')
print(f'  RETAINED with Y >= 0       {n_pos_p:>9,}   ({n_pos_p / max(int(p["usable"].sum()), 1):.2%})')


# ── CELL 9 : PANEL CHECKS ─────────────────────────────────────────────────────
print('\n' + '=' * 110)
print('CHECKS — PANEL')
print('=' * 110)

# (a) which shift convention is right? Test both against an independent
#     reconstruction. The market check compared against a separate column; here
#     there is none, so the two hypotheses are compared head to head. A
#     correlation-based leak test cannot settle this: it presupposes the
#     convention it is meant to verify.
print('  [a] shift convention test (max abs diff vs stored target, gap-free rows)')
for off, name in [(1, 'contemporaneous (dlyret[t] = day t)'),
                  (0, 'pre-shifted     (dlyret[t] = day t+1)')]:
    f = pd.concat([p.groupby('seg_id')['ret'].shift(-(k + off)) for k in range(5)], axis=1)
    ok = f.notna().all(axis=1)
    print(f'      {name:<42s} {np.nanmax((f.min(axis=1) - p["Y"])[ok].abs()):.2e}')
print('      The smaller value identifies the convention actually in force.')

# (b) leak test, meaningful given (a)
p['r_next'] = p.groupby('seg_id')['ret'].shift(-1)
sub_p = p['usable'] & p['r_next'].notna()
worst_p = max(abs(np.corrcoef(p.loc[sub_p, c], p.loc[sub_p, 'r_next'])[0, 1]) for c in REG)
print(f'\n  [b] max |corr(regressor, r_t+1)|, pooled = {worst_p:.4f}'
      f'{"                          OK" if worst_p < 0.20 else "   <-- INVESTIGATE"}')

# (c) reconstruction, segment-aware and restricted to gap-free windows
d_p = (p_chk - p['Y']).abs()[p['fwd_ok']]
mx_p = np.nanmax(d_p)
print(f'  [c] reconstruction of Y (gap-free windows, n={int(p["fwd_ok"].sum()):,}):'
      f' max abs diff = {mx_p:.2e}   {"OK" if mx_p < 1e-8 else "<-- MISMATCH"}')

# (d) RV22 backward-looking, spot check within a single segment
_sid = p.loc[p.index[p['usable']][500], 'seg_id']
_seg = p[p['seg_id'] == _sid].reset_index(drop=True)
_i = _seg.index[_seg['usable']][0]
_manual = (_seg['ret'].iloc[_i - 21:_i + 1] ** 2).sum()
print(f'  [d] RV22 in segment {_sid} at local row {_i}: manual {_manual:.6f} vs '
      f'stored {_seg["rv22"].iloc[_i]:.6f}   '
      f'{"OK" if abs(_manual - _seg["rv22"].iloc[_i]) < 1e-8 else "<-- MISMATCH"}')

PANEL_OK = (worst_p < 0.20) and (mx_p < 1e-8)
if not PANEL_OK:
    print('\n  *** PANEL ALIGNMENT CHECK FAILED — do not read the results below ***')


# ── CELL 10 : PANEL RESULTS ───────────────────────────────────────────────────
res_p = pd.DataFrame([r for s, c in SPLITS.items() for mn, cols in MODELS.items()
                      if (r := fit_panel(p, cols, c, s, mn)) is not None])

tbl_p = headline(res_p,
                 'TABLE 2 — PANEL.  Full model (L3), level space, stock fixed effects.',
                 extra=('n_stocks',))
piv_p = ladder(res_p, 'PANEL — incremental R^2 (train-mean benchmark)')


# ── CELL 11 : THE FLOOR ───────────────────────────────────────────────────────
print('\n' + '=' * 110)
print('THE FLOOR — subtract from the KAN')
print('=' * 110)
floor = pd.DataFrame({
    'market_L1': piv_m['L1'], 'market_L3': piv_m['L3'],
    'panel_L1':  piv_p['L1'], 'panel_L3':  piv_p['L3'],
    'market_MSE_L3': tbl_m.set_index('split')['MSE'],
    'panel_MSE_L3':  tbl_p.set_index('split')['MSE'],
})
print(floor.to_string(float_format=lambda v: f'{v:+.4f}'))

res_m.to_csv(RESULTS_DIR / 'baseline_level_market.csv', index=False)
res_p.to_csv(RESULTS_DIR / 'baseline_level_panel.csv', index=False)
print(f'\n  saved -> {RESULTS_DIR / "baseline_level_market.csv"}')
print(f'          {RESULTS_DIR / "baseline_level_panel.csv"}')

print(f"""
  READING THESE
  -------------
  Checks passed: market {MARKET_OK}, panel {PANEL_OK}. If either is False,
  nothing in that section should be quoted until it is resolved.

  Coefficients:  a0 = sig       sqrt(RV22)                     volatility scale
                 a1 = sig_acc   sqrt(RV22) * log(RV5/RV22)     scale x acceleration
                 a2 = sig_asym  sqrt(RV22) * (RSneg/RV5)       scale x asymmetry
  Cluster-robust t by calendar month in brackets. a0 should sit near -0.25.

  COMPARE ON MSE. It is convention-free, in the same units as the KAN's target,
  computed on the same rows, under the same squared-error loss the KAN
  minimises. No transformation is involved anywhere in this specification.

  R2 uses the test-period mean (sklearn r2_score convention). R2_trainmean is
  the honest out-of-sample statistic, since a forecaster standing at the start
  of the test period does not know the test mean. Report var_Y alongside both:
  across regime-split test periods R^2 measures test-period variance as much as
  model quality, and a calm period can produce the lowest MSE and the worst R^2
  simultaneously.

  L1 is the floor: volatility scale alone. L3-L1 is the shape contribution —
  how much acceleration and downside asymmetry add beyond the level, and the
  evidence that non-volatility information in the tail exists at all.
""")

LEVEL-SPACE STRUCTURAL BASELINE
  targets    : ..\..\..\Data\Data_Collection\Final\Stage_5_Model_Ready\03_targets
  splits     : ['Split_A', 'Split_B', 'Split_C', 'Split_D']   (5-day embargo at end of train)
  models     : L1 (1),  L2 (2),  L3 (3)   [full model = L3]
  gap policy : segments cut at calendar gaps > 10 days
  model      : Y = a0*sqrt(RV22) + a1*sqrt(RV22)*acc + a2*sqrt(RV22)*asym

MARKET — BUILD
  rows                                 4,379
  after 22-day warm-up                   4,357
  usable                               4,352
  dropped, incomplete forward              5
  zero-variance rows               RV22 0   RV5 0
  RETAINED with Y >= 0                   185   (4.25%)
    (a log specification must drop these; the level model does not,
     which removes selection on the dependent variable and puts the
     baseline on exactly the rows the KAN is scored on)

CHECKS — MARKET
  [a] ret[t] == target_daily_return[t-1] * 100                              OK
  [b] max |c

In [4]:
# ── CELL 11.5 : BACKTEST (MARKET, L3) ───────────────────────────────────────────
#
# Cells 6-7 above use ONLY masks()'s two-way split (embargoed train / test) --
# no validation set, because R2 and MSE need none: there is nothing to tune.
#
# The backtest is different. run_full_backtest picks a cash/invested threshold
# (and risk-scaled parameters) by searching for the best net-of-cost Sortino on
# one dataset, then applies that FIXED choice on a separate dataset. Searching
# and scoring on the same test data would inflate the reported Sharpe/Sortino
# by the same in-sample selection bias early-stopping-on-test would cause for a
# neural net. A validation slice is therefore needed for the threshold search
# only -- not for the model itself.
#
# OPTION B: the val tail is carved from INSIDE the same embargoed training
# window masks() already produces, purely to give the threshold search
# somewhere to look that isn't test. The coefficients used for both the val
# and test predictions are the SAME beta already reported in Table 1 -- fit on
# the full embargoed training window, nothing withheld from estimation. Only
# the threshold search is restricted to the val tail; the fitted model itself
# is identical to the one behind every number in Cells 6-7.

from evaluation import run_full_backtest
import json

VAL_DAYS = 252   # ~1 trading year, carved from the END of the embargoed
                 # training window, used ONLY to pick the backtest threshold


def fit_and_split_for_backtest(df, cols, cfg, val_days=VAL_DAYS, intercept=True):
    """
    Refit on masks()'s FULL embargoed training window -- identical beta to
    fit_market's Table 1 result, since it is the same OLS on the same X, y.
    Then predict on:
      - a VAL tail carved from inside that same training window (threshold
        search only, never used for estimation)
      - the same TEST window Table 1 already scores on
    """
    tr_mask, te_mask = masks(df, cfg)
    tr, te = df[tr_mask], df[te_mask]
    if len(tr) < 50 or len(te) < 20:
        return None

    base = lambda n: [np.ones(n)] if intercept else []
    Xtr = np.column_stack(base(len(tr)) + [tr[c].to_numpy() for c in cols])
    Xte = np.column_stack(base(len(te)) + [te[c].to_numpy() for c in cols])
    beta, _ = ols(Xtr, tr['Y'].to_numpy())          # == fit_market's beta exactly

    # Carve the val tail from WITHIN the training rows already selected by
    # masks() -- not a new/different date range, just the last val_days
    # trading dates of the window already used to fit beta.
    tr_dates_sorted = np.sort(tr['date'].unique())
    if len(tr_dates_sorted) > val_days:
        val_dates = tr_dates_sorted[-val_days:]
    else:
        val_dates = tr_dates_sorted
    val_row_mask = tr['date'].isin(val_dates)
    val = tr[val_row_mask]
    Xval = np.column_stack(base(len(val)) + [val[c].to_numpy() for c in cols])

    return {
        'val_dates':    val['date'].reset_index(drop=True),
        'val_yhat':     Xval @ beta,
        'val_returns':  val['target_daily_return'].to_numpy(),
        'test_dates':   te['date'].reset_index(drop=True),
        'test_yhat':    Xte @ beta,
        'test_returns': te['target_daily_return'].to_numpy(),
    }


print('\n' + '=' * 110)
print('BACKTEST — MARKET, L3, signal = predicted minret_5d_pct')
print('  beta identical to Table 1 (fit on the full embargoed training window).')
print('  Val tail used ONLY for threshold search, carved from inside that same')
print('  training window -- not a new fit, not new data.')
print('  go_cash_when="below": more negative predicted 5-day minimum => higher')
print('  crash risk => go to cash. Same convention as KAN/Ridge/Polymodel.')
print('=' * 110)

backtest_rows_m    = []
backtest_records_m = {}

for split, cfg in SPLITS.items():
    pred = fit_and_split_for_backtest(m, REG, cfg)
    if pred is None:
        print(f'  {split}: insufficient rows, skipped')
        continue

    bt = run_full_backtest(
        val_returns=pred['val_returns'],   val_signal=pred['val_yhat'],
        test_returns=pred['test_returns'], test_signal=pred['test_yhat'],
        go_cash_when='below',
        model_name='volatility_baseline_market',
        split_name=split,
    )
    backtest_records_m[split] = bt

    for strategy_name in ['simple', 'risk_scaled']:
        r = bt[strategy_name]
        backtest_rows_m.append({
            'split': split, 'strategy': strategy_name,
            'sharpe': r['sharpe'], 'sortino': r['sortino'],
            'annual_return': r['annual_return'], 'max_drawdown': r['max_drawdown'],
            'cumulative_return': r['cumulative_return'],
            'avg_exposure': r['avg_exposure'], 'annual_turnover': r['annual_turnover'],
            'buy_hold_sharpe': r['buy_hold_sharpe'], 'buy_hold_sortino': r['buy_hold_sortino'],
            'buy_hold_cumulative': r['buy_hold_cumulative'],
        })

pd.DataFrame(backtest_rows_m).to_csv(RESULTS_DIR / 'baseline_level_market_backtest_summary.csv', index=False)

def _json_safe(o):
    if isinstance(o, dict): return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, pd.DataFrame): return o.reset_index().to_dict(orient='records')
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return o

with open(RESULTS_DIR / 'baseline_level_market_backtest_full.json', 'w') as f:
    json.dump(_json_safe(backtest_records_m), f, indent=2, default=str)

print(f'\n  saved -> {RESULTS_DIR / "baseline_level_market_backtest_summary.csv"}')
print(f'          {RESULTS_DIR / "baseline_level_market_backtest_full.json"}')


BACKTEST — MARKET, L3, signal = predicted minret_5d_pct
  beta identical to Table 1 (fit on the full embargoed training window).
  Val tail used ONLY for threshold search, carved from inside that same
  training window -- not a new fit, not new data.
  go_cash_when="below": more negative predicted 5-day minimum => higher
  crash risk => go to cash. Same convention as KAN/Ridge/Polymodel.

════════════════════════════════════════════════════════════════════════
  BACKTEST  volatility_baseline_market  Split_A
════════════════════════════════════════════════════════════════════════

  Signal diagnostics (EMA span=5):
    Val  raw : [-2.587, -0.316]   smoothed: [-2.290, -0.544]
    Test raw : [-2.660, -0.254]   smoothed: [-2.442, -0.433]

  ── Simple Binary Timing  (cost=3.0bps, EMA=5) ──
  Threshold chosen on val (net-of-cost Sortino): -2.25

    Sharpe  Sortino   AnnRet    MaxDD    %Inv  Turn/yr   BH Shrp   BH Srt
  -----------------------------------------------------------------------

In [5]:
# ── CELL 12 : DOES ANY THEME ADD TO THE VOLATILITY BASELINE? ─────────────────
#
# One metric: change in out-of-sample R^2 from adding a theme's factors to the
# L3 volatility control.
#
#     dR2 = R2(control + theme) - R2(control)
#
# Two-stage, per split: fit the control by OLS on train; ridge the theme's
# factors onto the TRAIN RESIDUAL with the penalty chosen on validation; the
# final prediction is control + theme, scored against Y on test. Positive dR2
# means the theme carries information the volatility control does not.
#
# Out-of-sample throughout, so this answers "would it have helped", not
# "does it correlate". Reported per split so regime dependence is visible.
#
# TAXONOMY: uses the MEANS-ONLY numbered taxonomy, not the full-moments one.
# This cell analyses agg_means.parquet, whose stock-level columns carry bare
# base_factor names (Tax, not Tax_cwmean) with no moment suffix at all -- the
# means-only file has exactly one row per base_factor by construction, so
# there is no moment-vs-subtheme ambiguity here the way there is for
# agg_full_moments (see Group 3, which needs a different merge strategy for
# exactly this reason).
STAGE5   = TGT.parent
FEATURES = STAGE5 / '02_assembled' / 'agg_means.parquet'
THEMES   = STAGE5 / '05_themes'    / 'numbered_classified_moment_inventory_means_only.csv'
ALPHAS   = [10, 100, 1_000, 10_000, 100_000]

feat = pd.read_parquet(FEATURES).drop(columns=['target_daily_return'], errors='ignore')
fcols = [x for x in feat.columns if x != 'date']

D = (m.loc[m['usable'], ['date', 'Y'] + REG]
       .merge(feat, on='date', how='inner')
       .reset_index(drop=True))

th = pd.read_csv(THEMES, dtype={'theme_id': str, 'subtheme_id': str})[
    ['base_factor', 'theme_name']
].drop_duplicates('base_factor')
th = th[th['theme_name'].notna()]                    # drop unassigned factors
theme_of = dict(zip(th['base_factor'], th['theme_name'].astype(str)))
themes = sorted({theme_of[c] for c in fcols if c in theme_of})
THEME_COLS = {t: [c for c in fcols if theme_of.get(c) == t] for t in themes}

n_unassigned = len([c for c in fcols if c not in theme_of])
print(f'  unassigned factors excluded: {n_unassigned}')

print('=' * 92)
print('DOES ANY THEME ADD TO THE VOLATILITY BASELINE?')
print('=' * 92)
print(f'  rows {len(D):,}   factors {len(fcols)}   themes {len(THEME_COLS)}')
print(f'  control : Y ~ {" + ".join(REG)}   (fitted on train, per split)')
print(f'  metric  : out-of-sample dR2 on test, train-mean benchmark')


def masks3(df, cfg):
    """Train (with embargo) / validation (the gap) / test."""
    d = df['date']
    tr_dates = np.sort(d[d <= cfg['train_end']].unique())
    keep = tr_dates[:-EMBARGO_DAYS] if len(tr_dates) > EMBARGO_DAYS else tr_dates
    return (d.isin(keep).to_numpy(),
            ((d > cfg['train_end']) & (d < cfg['test_start'])).to_numpy(),
            ((d >= cfg['test_start']) & (d <= cfg['test_end'])).to_numpy())


def ridge_fit(Ztr, rtr, Zva, rva, alphas):
    """Ridge on standardised, centred data. Penalty chosen on validation SSE."""
    G = Ztr.T @ Ztr
    best_w, best_sse = None, np.inf
    for a in alphas:
        w = np.linalg.solve(G + a * np.eye(G.shape[0]), Ztr.T @ rtr)
        sse = np.sum((rva - Zva @ w) ** 2)
        if sse < best_sse:
            best_w, best_sse = w, sse
    return best_w


rows = []
for split, cfg in SPLITS.items():
    tr, va, te = masks3(D, cfg)
    y = D['Y'].to_numpy(float)
    ybar = y[tr].mean()

    # ── control, OLS on train ──
    C = np.column_stack([np.ones(len(D))] + [D[c].to_numpy(float) for c in REG])
    b, _ = ols(C[tr], y[tr])
    ctrl = C @ b
    r2_ctrl = r2(y[te], ctrl[te], ybar)
    resid = y - ctrl                      # what the control misses, all rows

    for theme, cols in THEME_COLS.items():
        Xf = D[cols].to_numpy(float)
        # train-median impute so the row set is identical for every theme
        med = np.nanmedian(Xf[tr], axis=0)
        Xf = np.where(np.isnan(Xf), med, Xf)
        mu, sd = Xf[tr].mean(0), Xf[tr].std(0)
        sd[sd == 0] = 1.0
        Z = (Xf - mu) / sd
        rbar = resid[tr].mean()
        w = ridge_fit(Z[tr], resid[tr] - rbar, Z[va], resid[va] - rbar, ALPHAS)
        yhat = ctrl + rbar + Z @ w        # control + theme
        rows.append({'split': split, 'theme': theme, 'n_feat': len(cols),
                     'R2_ctrl': r2_ctrl,
                     'R2_both': r2(y[te], yhat[te], ybar),
                     'dR2': r2(y[te], yhat[te], ybar) - r2_ctrl})

res = pd.DataFrame(rows)

# ── the control's own out-of-sample R2, for reference ──
print('\n  control R2 out-of-sample, by split:')
print('   ' + '   '.join(f'{s} {v:+.4f}' for s, v in
                         res.groupby('split')['R2_ctrl'].first().items()))

# ── the table ──
piv = res.pivot(index='theme', columns='split', values='dR2')
piv['mean'] = piv.mean(axis=1)
piv['n'] = res.groupby('theme')['n_feat'].first()
piv = piv.sort_values('mean', ascending=False)

print('\n' + '=' * 92)
print('CHANGE IN OUT-OF-SAMPLE R^2 FROM ADDING EACH THEME')
print('=' * 92 + '\n')
print(piv[['n'] + list(SPLITS) + ['mean']].to_string(
    formatters={c: '{:+.4f}'.format for c in list(SPLITS) + ['mean']}))
print('\n  Positive = the theme improves on the volatility control.')
print('  Negative = adding it makes the forecast worse out-of-sample.')

res.to_csv(RESULTS_DIR / 'theme_beyond_volatility_oos.csv', index=False)
print(f'\n  saved -> {RESULTS_DIR / "theme_beyond_volatility_oos.csv"}')


# ── CELL 13 : SAME TEST AT SUBTHEME AND FACTOR LEVEL ─────────────────────────
#
# Identical machinery to Cell 12, so the three tables are directly comparable.
# Adds n_pos: in how many of the 4 splits dR2 is positive. With means this noisy,
# consistency across splits is more informative than the mean itself.

def eval_groups(groups, D):
    rows = []
    for split, cfg in SPLITS.items():
        tr, va, te = masks3(D, cfg)
        y = D['Y'].to_numpy(float)
        ybar = y[tr].mean()
        C = np.column_stack([np.ones(len(D))] + [D[c].to_numpy(float) for c in REG])
        b, _ = ols(C[tr], y[tr])
        ctrl = C @ b
        r2_ctrl = r2(y[te], ctrl[te], ybar)
        resid = y - ctrl
        for name, cols in groups.items():
            Xf = D[cols].to_numpy(float)
            med = np.nanmedian(Xf[tr], axis=0)
            Xf = np.where(np.isnan(Xf), med, Xf)
            Xf = np.nan_to_num(Xf, nan=0.0)
            mu, sd = Xf[tr].mean(0), Xf[tr].std(0)
            sd = np.where(~np.isfinite(sd) | (sd == 0), 1.0, sd)
            Z = (Xf - mu) / sd
            rbar = resid[tr].mean()
            w = ridge_fit(Z[tr], resid[tr] - rbar, Z[va], resid[va] - rbar, ALPHAS)
            yhat = ctrl + rbar + Z @ w
            rows.append({'split': split, 'name': name, 'n_feat': len(cols),
                         'dR2': r2(y[te], yhat[te], ybar) - r2_ctrl})
    return pd.DataFrame(rows)


def table(res, label, extra=None, top=None):
    piv = res.pivot(index='name', columns='split', values='dR2')
    piv['mean'] = piv.mean(axis=1)
    piv['n_pos'] = (piv[list(SPLITS)] > 0).sum(axis=1)
    piv['n'] = res.groupby('name')['n_feat'].first()
    if extra is not None:
        piv['theme'] = piv.index.map(extra)
    piv = piv.sort_values('mean', ascending=False)
    if top:
        piv = piv.head(top)
    cols = (['n'] + (['theme'] if extra is not None else [])
            + list(SPLITS) + ['mean', 'n_pos'])
    print('\n' + '=' * 118)
    print(label)
    print('=' * 118 + '\n')
    print(piv[cols].to_string(
        formatters={c: '{:+.4f}'.format for c in list(SPLITS) + ['mean']}))
    return piv


# ── build subtheme and factor groupings ──
sub_raw = (pd.read_csv(THEMES, dtype={'theme_id': str, 'subtheme_id': str})[
    ['base_factor', 'subtheme_name', 'theme_name']
].drop_duplicates('base_factor'))
sub_raw = sub_raw[sub_raw['subtheme_name'].notna() & sub_raw['theme_name'].notna()]
sub_of = dict(zip(sub_raw['base_factor'], sub_raw['subtheme_name'].astype(str)))
sub_parent = dict(zip(sub_raw['subtheme_name'].astype(str),
                      sub_raw['theme_name'].astype(str)))

SUB_COLS = {}
for c in fcols:
    if c in sub_of:
        SUB_COLS.setdefault(sub_of[c], []).append(c)
SUB_COLS = {k: v for k, v in sorted(SUB_COLS.items())}

FACTOR_COLS = {c: [c] for c in fcols if c in theme_of}

print(f'  subthemes {len(SUB_COLS)}   factors {len(FACTOR_COLS)}')

# ── run ──
res_sub = eval_groups(SUB_COLS, D)
res_fac = eval_groups(FACTOR_COLS, D)

tbl_sub = table(res_sub, 'SUBTHEMES — change in out-of-sample R^2 (all, sorted by mean)',
                extra=sub_parent)
tbl_fac = table(res_fac, 'TOP 50 FACTORS — change in out-of-sample R^2',
                extra=theme_of, top=50)

print('\n  n_pos = number of splits (of 4) where dR2 > 0.')
print('  Compare each row\'s mean against its spread across splits. Where the')
print('  spread exceeds the mean, the mean is not evidence of anything.')

res_sub.to_csv(RESULTS_DIR / 'subtheme_beyond_volatility_oos.csv', index=False)
res_fac.to_csv(RESULTS_DIR / 'factor_beyond_volatility_oos.csv', index=False)
print(f'\n  saved -> {RESULTS_DIR / "subtheme_beyond_volatility_oos.csv"}')
print(f'          {RESULTS_DIR / "factor_beyond_volatility_oos.csv"}')


# ── CELL 14 : SEARCH PENALTY — MAX |CORR| AGAINST THE BASELINE RESIDUAL ──────
#
# The claim being tested: no individual factor's linear correlation with what the
# volatility baseline misses exceeds what chance alone produces across a search
# of this width.
#
# Everything is computed on the TRAINING window only, per split. That is the
# honest version of the test — a researcher screening factors would have had
# access to train, not test — and it differs from the earlier full-sample
# diagnostic, which included the evaluation periods.
#
# Under a global null of no relationship, the largest absolute correlation over
# n0 candidates on N_eff independent observations is approximately
#
#       max_p |rho_p|  ~  sqrt( 2 ln(n0) / N_eff )
#
# N_eff is well below the row count: consecutive five-day forward windows share
# four of their five days, giving a variance inflation of
# 1 + 2*sum_{k=1..4}(1 - k/5) = 5. Persistent regressors inflate it further, so
# n_train/5 is itself an upper bound on N_eff and therefore a LOWER bound on the
# floor. Both are reported.
#
# This is a LINEAR screen. Its nonlinear counterpart is the per-factor univariate
# LNLM ranking in the polymodel notebook.
#
# Depends on m, REG, ols, masks, SPLITS, EMBARGO_DAYS, TGT, RESULTS_DIR from
# earlier cells in this notebook. Uses the same means-only agg_means.parquet as
# Cells 12-13 -- this is a factor-level screen, so the taxonomy itself is not
# needed here, only the feature matrix.
STAGE5   = TGT.parent
FEATURES = STAGE5 / '02_assembled' / 'agg_means.parquet'
OVERLAP_INFLATION = 5.0      # from the 5-day target overlap alone


def noise_floor(n0, n_eff):
    """Expected max |correlation| under a global null over n0 candidates."""
    return np.sqrt(2 * np.log(n0) / n_eff)


# ── assemble features alongside the target and the control regressors ─────────
feat = (pd.read_parquet(FEATURES)
          .drop(columns=['target_daily_return'], errors='ignore'))
feat['date'] = pd.to_datetime(feat['date'])
fcols = [c for c in feat.columns if c != 'date']

D = (m.loc[m['usable'], ['date', 'Y'] + REG]
       .merge(feat, on='date', how='inner')
       .reset_index(drop=True))

# rows where every factor is present, so the search is over a constant candidate
# set rather than a set that shrinks with missingness
D = D[D[fcols].notna().all(axis=1) & np.isfinite(D[fcols].to_numpy()).all(axis=1)]
D = D.reset_index(drop=True)
n0 = len(fcols)

print('=' * 92)
print('SEARCH PENALTY — MAX |CORR| WITH THE BASELINE RESIDUAL, TRAINING WINDOW')
print('=' * 92)
print(f'  candidate factors n0   {n0:,}')
print(f'  rows available          {len(D):,}   '
      f'{D["date"].min().date()} .. {D["date"].max().date()}')
print(f'  control                 Y ~ {" + ".join(REG)}   (refitted per split)')


def abs_corr(v, F):
    """|Pearson correlation| of every column of F with the vector v."""
    vc, Fc = v - v.mean(), F - F.mean(axis=0)
    den = np.sqrt((Fc ** 2).sum(axis=0) * (vc ** 2).sum())
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.abs(np.where(den > 0, (Fc.T @ vc) / den, 0.0))


rows, per_factor = [], {}
for split, cfg in SPLITS.items():
    d = D['date']
    tr_dates = np.sort(d[d <= cfg['train_end']].unique())
    keep = tr_dates[:-EMBARGO_DAYS] if len(tr_dates) > EMBARGO_DAYS else tr_dates
    tr = d.isin(keep).to_numpy()

    y = D.loc[tr, 'Y'].to_numpy(float)
    X = np.column_stack([np.ones(tr.sum())]
                        + [D.loc[tr, c].to_numpy(float) for c in REG])
    _, e = ols(X, y)                                  # training residual

    F = D.loc[tr, fcols].to_numpy(float)
    r = abs_corr(e, F)
    per_factor[split] = pd.Series(r, index=fcols)

    n_train = int(tr.sum())
    n_eff   = n_train / OVERLAP_INFLATION
    floor   = noise_floor(n0, n_eff)

    rows.append({
        'split': split, 'n_train': n_train, 'n_eff': n_eff,
        'floor': floor,
        'max_corr': r.max(), 'max_r2': r.max() ** 2,
        'best_factor': fcols[int(np.argmax(r))],
        'n_above': int((r > floor).sum()),
        'ctrl_r2_train': 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2),
    })

res = pd.DataFrame(rows)

print('\n  PER SPLIT')
print('  ' + '-' * 88)
print(res[['split', 'n_train', 'n_eff', 'ctrl_r2_train',
           'max_corr', 'max_r2', 'floor', 'n_above', 'best_factor']]
      .to_string(index=False, formatters={
          'n_train': '{:,}'.format, 'n_eff': '{:,.0f}'.format,
          'ctrl_r2_train': '{:+.4f}'.format, 'max_corr': '{:.4f}'.format,
          'max_r2': '{:.4f}'.format, 'floor': '{:.4f}'.format}))

print(f"\n  max |corr| across all splits   {res['max_corr'].max():.4f}"
      f"   (floor {res['floor'].mean():.4f})")
print(f"  splits where max |corr| exceeds the floor   "
      f"{(res['max_corr'] > res['floor']).sum()} of {len(res)}")

# ── strongest factors, and their consistency across splits ───────────────────
P = pd.DataFrame(per_factor)
P['mean'] = P[list(SPLITS)].mean(axis=1)
P['n_above'] = (P[list(SPLITS)].gt(res.set_index('split')['floor'])).sum(axis=1)

print('\n  TOP 15 FACTORS BY MEAN |CORR| ACROSS SPLITS')
print('  ' + '-' * 88)
print(P.nlargest(15, 'mean')[list(SPLITS) + ['mean', 'n_above']]
      .to_string(formatters={c: '{:.4f}'.format for c in list(SPLITS) + ['mean']}))
print('\n  n_above = splits (of 4) where that factor exceeds its own split floor.')

# ── how the conclusion depends on the N_eff assumption ───────────────────────
print('\n  SENSITIVITY TO N_eff')
print('  ' + '-' * 88)
print(f"  {'N_eff':>8s} {'floor':>8s} {'verdict vs observed max':>34s}")
obs = res['max_corr'].max()
for n_eff in (250, 500, 750, 1000, int(res['n_train'].mean())):
    f = noise_floor(n0, n_eff)
    verdict = 'observed max EXCEEDS floor' if obs > f else 'observed max below floor'
    print(f"  {n_eff:8,d} {f:8.4f} {verdict:>34s}")
print("""
  The last row uses the raw row count, which ignores the five-day overlap and is
  therefore the most permissive floor possible. If the observed maximum sits
  below the floor even there, no assumption about N_eff can rescue a
  factor-by-factor search.
""")

P.to_csv(RESULTS_DIR / 'search_penalty_train_corr.csv')
print(f'  saved -> {RESULTS_DIR / "search_penalty_train_corr.csv"}')

  unassigned factors excluded: 0
DOES ANY THEME ADD TO THE VOLATILITY BASELINE?
  rows 4,352   factors 574   themes 13
  control : Y ~ sig + sig_acc + sig_asym   (fitted on train, per split)
  metric  : out-of-sample dR2 on test, train-mean benchmark

  control R2 out-of-sample, by split:
   Split_A +0.1806   Split_B +0.2611   Split_C +0.2901   Split_D +0.0228

CHANGE IN OUT-OF-SAMPLE R^2 FROM ADDING EACH THEME

split                              n Split_A Split_B Split_C Split_D    mean
theme                                                                       
Interest Rates & Monetary Policy  57 +0.0176 -0.0020 +0.0031 +0.0056 +0.0061
Valuation                         18 +0.0033 -0.0012 -0.0019 +0.0057 +0.0015
Liquidity & Market Quality        41 +0.0048 -0.0044 +0.0022 +0.0008 +0.0009
Cross-Sectional Risk Profile      20 -0.0009 +0.0044 +0.0004 -0.0005 +0.0008
Momentum & Reversal               40 +0.0040 -0.0040 +0.0056 -0.0049 +0.0001
Credit Conditions                 24 +0.0037 

### Measuring $c$: the coefficient of variation of volatility

The volatility-only predictability ceiling has exactly one free parameter. Everything else
in the derivation is a constant that follows from the distribution of the minimum of five
standard normal draws:

$$R^2_{\max} = \frac{1.3525\,c^2}{1.8000\,c^2 + 0.4475}$$

where $1.3525 = \mathbb{E}[\min_5 z]^2$ and $1.8000 = \mathbb{E}[(\min_5 z)^2]$. The single
input is $c$.

#### What it is

The coefficient of variation is dispersion expressed as a fraction of the mean:

$$c \;=\; \frac{\text{sd}(\sigma_t)}{\mathbb{E}[\sigma_t]}$$

It answers: **how much does market volatility move around, relative to how volatile the
market typically is?** A market whose daily volatility sits permanently at 1% has $c = 0$
and nothing about the level is forecastable, so the ceiling is zero. A market swinging
between 0.5% and 4% has a large $c$ and a correspondingly higher ceiling.

The reason the ceiling depends on $c$ and not on the level of $\sigma$ is that the target
scales linearly with $\sigma$. A model can only explain variance in the target that comes
from variance in $\sigma$; the average level of $\sigma$ scales target and prediction
equally and therefore cancels. Being a ratio, $c$ is also unit-free — it is unchanged
whether volatility is measured in percent, in basis points, daily or annualised.

#### How it is measured here

Volatility is unobservable, so it is proxied by realised volatility over a trailing window:

$$\hat\sigma_t \;=\; \sqrt{\sum_{j=0}^{w-1} r_{t-j}^2}, \qquad c \;=\; \frac{\text{sd}(\hat\sigma_t)}{\text{mean}(\hat\sigma_t)}$$

Note that $\sqrt{\text{RV}_w} = \sqrt{w}\,\sigma_{\text{daily}}$, and the $\sqrt{w}$ is a
constant multiplier. Since $c$ is a ratio, it cancels — so the window length does not
affect $c$ through scaling. It affects it only through **measurement noise**.

#### The one thing to be careful about

Realised volatility is a noisy estimate of true conditional volatility, and noise adds
variance. So the measured dispersion decomposes as

$$\text{Var}(\hat\sigma) \;=\; \underbrace{\text{Var}(\sigma)}_{\text{what we want}} \;+\; \underbrace{\text{Var}(\text{estimation error})}_{\text{contamination}}$$

which means **every measured $c$ is biased upward, and short windows more so than long
ones.** A 22-day window contains roughly a tenth of the observations of a 252-day window
and is correspondingly noisier.

The code therefore reports three things: $c$ at three window lengths, so the direction and
size of the bias is visible; a noise-corrected estimate obtained from the autocorrelation
of $\log\hat\sigma$, which separates the persistent component from the day-to-day
estimation error; and $c$ per test period, since volatility is more dispersed in some
regimes than others and the ceiling is therefore regime-dependent.

The 22-day figure is the operative one — it matches the baseline's own window and is
closest to what a model at a five-day horizon could exploit — but it should be quoted as
an upper bound, with the corrected value alongside.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# MEASURE c — THE COEFFICIENT OF VARIATION OF VOLATILITY
#
# Standalone. Reads only the market target file. No taxonomy dependency, no
# shared state with the beyond-volatility cells (Group 1) or the main
# structural baseline above -- this can be run on its own, in any order,
# in the same notebook or a separate one.
#
# c = sd(sigma) / mean(sigma), with sigma proxied by sqrt(RV22).
#
# This is the single free parameter in the volatility-only predictability
# ceiling:
#       R2_max = 1.3525 c^2 / (1.8000 c^2 + 0.4475)
# derived from E[min of 5 standard normals] = -1.163 and E[M^2] = 1.8000.
#
# NOTE ON LOCAL SPLITS: this block's SPLITS is a (start, end) tuple keyed
# dict -- deliberately a different shape from the {'train_end', 'test_start',
# 'test_end'} dict used everywhere else in this notebook. If this cell is run
# in the same kernel session as the main baseline or Group 1 cells, this
# reassigns the name SPLITS for the rest of the session. Re-run the earlier
# cells' SPLITS definition again before re-running anything downstream of it
# if you execute cells out of order.
# ═══════════════════════════════════════════════════════════════════════════════

from pathlib import Path
import numpy as np
import pandas as pd

TGT = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets')
RESULTS_DIR = Path('../../../Data/Results/Dense_vs_Sparse_KAN/Volatility_Baseline')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = {
    'Split_A': ('2018-01-01', '2019-12-31'),
    'Split_B': ('2020-01-01', '2021-12-31'),
    'Split_C': ('2022-01-01', '2023-12-31'),
    'Split_D': ('2023-01-01', '2024-12-31'),
}


def ceiling(c):
    """Volatility-only predictability ceiling as a function of c."""
    return 1.3525 * c**2 / (1.8000 * c**2 + 0.4475)


# ── build sigma ────────────────────────────────────────────────────────────────
m = (pd.read_parquet(TGT / 'targets_market.parquet')
       .sort_values('date').reset_index(drop=True))
m['date'] = pd.to_datetime(m['date'])

# target_daily_return is pre-shifted (row t holds day t+1's return), so unshift
# once to get the return realised ON day t before building trailing windows
m['ret'] = m['target_daily_return'].astype('float64').shift(1) * 100.0

# sqrt(RV22) is sqrt(22) * sigma_daily; the constant cancels in a coefficient
# of variation, so c is unaffected by the window scaling
sq = m['ret'] ** 2
for w in (22, 63, 252):
    m[f'sig{w}'] = np.sqrt(sq.rolling(w).sum())

print('=' * 74)
print('COEFFICIENT OF VARIATION OF VOLATILITY')
print('=' * 74)
print(f"  {len(m):,} rows   {m['date'].min().date()} .. {m['date'].max().date()}")
print(f"\n  ceiling formula:  R2_max = 1.3525 c^2 / (1.8000 c^2 + 0.4475)")

# ── full sample, three window lengths ─────────────────────────────────────────
full_rows = []
print(f"\n  FULL SAMPLE")
print(f"  {'window':>8s} {'n':>7s} {'mean sig':>10s} {'sd sig':>9s} "
      f"{'c':>7s} {'R2_max':>8s}")
print("  " + "-" * 54)
for w in (22, 63, 252):
    s = m[f'sig{w}'].dropna()
    c = s.std() / s.mean()
    full_rows.append({'window': w, 'n': len(s), 'mean_sig': s.mean(),
                      'sd_sig': s.std(), 'c': c, 'R2_max': ceiling(c)})
    print(f"  {w:6d}d {len(s):7,d} {s.mean():10.3f} {s.std():9.3f} "
          f"{c:7.3f} {ceiling(c):8.3f}")

print("\n  Longer windows give a less noisy estimate of sigma. Because measurement")
print("  noise inflates the observed variance, c from a short window is an upper")
print("  bound and the 22-day figure therefore overstates the ceiling slightly.")

# ── per test period, using the 22-day proxy ───────────────────────────────────
period_rows = []
print(f"\n  BY TEST PERIOD  (sqrt(RV22))")
print(f"  {'split':>9s} {'period':>12s} {'n':>6s} {'mean sig':>10s} "
      f"{'c':>7s} {'R2_max':>8s}")
print("  " + "-" * 56)
for name, (a, b) in SPLITS.items():
    s = m.loc[(m['date'] >= a) & (m['date'] <= b), 'sig22'].dropna()
    c = s.std() / s.mean()
    period_rows.append({'split': name, 'period_start': a, 'period_end': b,
                        'n': len(s), 'mean_sig': s.mean(),
                        'sd_sig': s.std(), 'c': c, 'R2_max': ceiling(c)})
    print(f"  {name:>9s} {a[:4]+'-'+b[:4]:>12s} {len(s):6,d} "
          f"{s.mean():10.3f} {c:7.3f} {ceiling(c):8.3f}")

print("\n  c varies across periods because volatility itself is more dispersed in")
print("  some regimes than others. The ceiling is therefore regime-dependent, and")
print("  a calm test period has a genuinely lower achievable R2 than a turbulent")
print("  one — independent of any model.")

# ── sensitivity, for the write-up table ───────────────────────────────────────
sens_rows = []
print(f"\n  SENSITIVITY")
print(f"  {'c':>7s} {'R2_max':>8s}")
print("  " + "-" * 16)
for c in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    sens_rows.append({'c': c, 'R2_max': ceiling(c)})
    print(f"  {c:7.1f} {ceiling(c):8.3f}")

# ── save ────────────────────────────────────────────────────────────────────
pd.DataFrame(full_rows).to_csv(RESULTS_DIR / 'coefficient_of_variation_full_sample.csv', index=False)
pd.DataFrame(period_rows).to_csv(RESULTS_DIR / 'coefficient_of_variation_by_period.csv', index=False)
pd.DataFrame(sens_rows).to_csv(RESULTS_DIR / 'coefficient_of_variation_sensitivity.csv', index=False)

print(f'\n  saved -> {RESULTS_DIR / "coefficient_of_variation_full_sample.csv"}')
print(f'          {RESULTS_DIR / "coefficient_of_variation_by_period.csv"}')
print(f'          {RESULTS_DIR / "coefficient_of_variation_sensitivity.csv"}')

COEFFICIENT OF VARIATION OF VOLATILITY
  4,379 rows   2007-08-01 .. 2024-12-20

  ceiling formula:  R2_max = 1.3525 c^2 / (1.8000 c^2 + 0.4475)

  FULL SAMPLE
    window       n   mean sig    sd sig       c   R2_max
  ------------------------------------------------------
      22d   4,357      4.894     3.298   0.674    0.486
      63d   4,316      8.589     5.132   0.598    0.443
     252d   4,127     18.240     8.451   0.463    0.348

  Longer windows give a less noisy estimate of sigma. Because measurement
  noise inflates the observed variance, c from a short window is an upper
  bound and the 22-day figure therefore overstates the ceiling slightly.

  BY TEST PERIOD  (sqrt(RV22))
      split       period      n   mean sig       c   R2_max
  --------------------------------------------------------
    Split_A    2018-2019    503      4.206   0.472    0.355
    Split_B    2020-2021    505      5.885   0.828    0.551
    Split_C    2022-2023    501      5.786   0.350    0.248
    Sp

In [1]:
import pandas as pd
sc = pd.read_csv(r"..\..\..\Data\Results\Dense_vs_Sparse_KAN\Polymodel\diagnostics\subtheme_composite.csv")
for fs, c in sc.groupby("feature_set"):
    u = c.drop_duplicates("subtheme_name")
    print(fs, "| subthemes", c.subtheme_name.nunique(), "| mean members", round(u.n.mean(),2),
          "| rho", round(c.rho_bar.median(),3), "| m_eff", round(c.m_eff.median(),2),
          "| amp_pred", round(c.amp_pred.median(),2), "| amp_obs", round(c.amp_obs.median(),2))

agg_full_moments | subthemes 284 | mean members 5.65 | rho 0.378 | m_eff 1.87 | amp_pred 1.37 | amp_obs 1.36
agg_means | subthemes 105 | mean members 5.03 | rho 0.428 | m_eff 1.74 | amp_pred 1.32 | amp_obs 1.25
